# RAG Inference Pipeline

## Overview

This notebook is **Part 2 of 3** in the RAGAS Evaluation Demo. It runs the synthetic questions (generated in notebook 1) through a real RAG system built on Llama Stack, so that the RAGAS evaluation in notebook 3 measures an actual retrieval-augmented generation pipeline.

**Prerequisites:**
- Complete **[`1.dataset_generation.ipynb`](1.dataset_generation.ipynb)** first to generate the synthetic evaluation dataset
- Ensure the `rag_evaluation_dataset.jsonl` file exists in the current directory

**Next:** After completing this notebook, proceed to **[`3.ragas-evaluation.ipynb`](3.ragas-evaluation.ipynb)** to evaluate the RAG answers using RAGAS metrics.

## What This Notebook Does

1. **Connect to Llama Stack** and discover available models
2. **Upload the source PDF** used for dataset generation
3. **Create a vector store** with chunked embeddings (Milvus-backed)
4. **Load synthetic questions** from notebook 1
5. **Run each question through the RAG pipeline** using `file_search`
6. **Save RAG answers and retrieved contexts** to `rag_inference_dataset.jsonl`

In [ ]:
%pip install -r requirements.txt

In [ ]:
import json
import os
from pathlib import Path
from typing import List

from llama_stack_client import LlamaStackClient

## Step 1: Client Setup & Model Discovery

Connect to the Llama Stack server and discover the available inference and embedding models.

In [ ]:
client = LlamaStackClient(
    base_url=os.getenv("URL", "http://lsd-ragas-example-service:8321"), timeout=600
)

models = client.models.list()

# Find the inference (LLM) model
inference_model = next(
    m
    for m in models
    if m.custom_metadata and m.custom_metadata.get("model_type") == "llm"
)
print(f"Inference model: {inference_model.id}")

# Find the embedding model
embedding_model = next(
    m
    for m in models
    if m.custom_metadata and m.custom_metadata.get("model_type") == "embedding"
)
embedding_dimension = int(
    float(embedding_model.custom_metadata.get("embedding_dimension"))
)
print(f"Embedding model: {embedding_model.id} (dimension: {embedding_dimension})")

## Step 2: Upload PDF

Upload the same source PDF used in notebook 1 so the RAG system can retrieve from it.

In [ ]:
pdf_path = "ibm-annual-report-2024.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"PDF file not found: {pdf_path}\nEnsure the PDF is in the current directory."
    )

with open(pdf_path, "rb") as f:
    uploaded_file = client.files.create(
        file=(pdf_path, f, "application/pdf"),
        purpose="assistants",
    )

print(f"Uploaded '{pdf_path}' (file_id: {uploaded_file.id})")

## Step 3: Create Vector Store

Create a Milvus-backed vector store, chunk the PDF, and embed it.

The two-step pattern (create empty store, then add files) avoids timeouts on large documents.

In [ ]:
vector_store_name = "ragas-eval-ibm-annual-report"

# Delete existing vector store with the same name for idempotent re-runs
for vs in client.vector_stores.list():
    if vs.name == vector_store_name:
        client.vector_stores.delete(vs.id)
        print(f"Deleted existing vector store: {vs.id}")

# Warm up the embedding model
client.embeddings.create(model=embedding_model.id, input="warmup")

# Create empty vector store with chunking config
vector_store = client.vector_stores.create(
    name=vector_store_name,
    file_ids=[],
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 512,
            "chunk_overlap_tokens": 64,
        },
    },
    extra_body={
        "embedding_model": embedding_model.id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "milvus",
    },
)
print(f"Created vector store '{vector_store_name}' (id: {vector_store.id})")

# Add file to vector store separately
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=uploaded_file.id,
)
vector_store = client.vector_stores.retrieve(vector_store.id)
print(f"Vector store ready: {vector_store}")

## Step 4: Load Synthetic Questions

Load the evaluation dataset produced by notebook 1. We keep the `ground_truth` and original `contexts` from the synthetic generation to use as reference during RAGAS evaluation.

In [ ]:
input_path = "rag_evaluation_dataset.jsonl"

if not os.path.exists(input_path):
    raise FileNotFoundError(
        f"Dataset not found: {input_path}\n"
        "Run notebook 1 (1.dataset_generation.ipynb) first."
    )

synthetic_data = []
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            synthetic_data.append(json.loads(line))

print(f"Loaded {len(synthetic_data)} questions from {input_path}")

## Step 5: Run RAG Inference

For each synthetic question, query the RAG system using Llama Stack's Responses API with the `file_search` tool. This retrieves relevant chunks from the vector store and generates an answer grounded in those chunks.

In [ ]:
def extract_retrieved_contexts(response) -> List[str]:
    """
    Extract retrieved contexts from a LlamaStack Responses API output.

    Args:
        response: Response object from client.responses.create()

    Returns:
        List of retrieved context strings
    """
    retrieved_contexts = []

    for output_item in response.output:
        if (
            hasattr(output_item, "type")
            and output_item.type == "file_search_call"
            and hasattr(output_item, "results")
            and output_item.results
        ):
            for result in output_item.results:
                if hasattr(result, "text") and result.text:
                    retrieved_contexts.append(result.text)

    return retrieved_contexts

In [ ]:
rag_results = []

for i, record in enumerate(synthetic_data):
    question = record["question"]
    print(f"[{i + 1}/{len(synthetic_data)}] {question[:80]}...")

    try:
        resp = client.responses.create(
            model=inference_model.id,
            instructions="""
                /no_think
                You are a helpful assistant with access to data via the file_search tool.

                When asked questions, use available tools to find the answer. Follow these rules:
                1. Use tools immediately without asking for confirmation
                2. Chain tool calls as needed
                3. Do not narrate your process
                4. Only provide the final answer
                5. If the answer is not found in the context, respond with 'I don't know'
            """,
            tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
            stream=False,
            input=question,
        )

        rag_answer = resp.output_text.strip()
        rag_contexts = extract_retrieved_contexts(resp)

    except Exception as e:
        print(f"  ERROR: {e}")
        rag_answer = ""
        rag_contexts = []

    rag_results.append(
        {
            "question": question,
            "answer": rag_answer,
            "contexts": rag_contexts if rag_contexts else [""],
            "ground_truth": record.get("ground_truth", ""),
            "ground_truth_contexts": record.get("contexts", [""]),
        }
    )

    print(f"  Answer: {rag_answer[:120]}...")
    print(f"  Retrieved {len(rag_contexts)} context(s)")

print(f"\nCompleted RAG inference for {len(rag_results)} questions")

## Step 6: Save RAG Inference Dataset

Save the RAG answers and retrieved contexts to a new JSONL file. This keeps notebook 1's `rag_evaluation_dataset.jsonl` untouched.

In [ ]:
output_path = Path("rag_inference_dataset.jsonl")

with output_path.open("w", encoding="utf-8") as f:
    for record in rag_results:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(rag_results)} records to {output_path}")

## Step 7: Cleanup (Optional)

Uncomment the cell below to delete the vector store when you no longer need it.

In [ ]:
# client.vector_stores.delete(vector_store.id)
# print(f"Deleted vector store: {vector_store.id}")

## Summary

You have successfully:

1. Uploaded the source PDF and created a vector store
2. Run each synthetic question through the RAG pipeline
3. Saved RAG answers and retrieved contexts to `rag_inference_dataset.jsonl`

### Generated Files

- `rag_inference_dataset.jsonl` - RAG answers with retrieved contexts, ready for RAGAS evaluation

### Next Steps

Proceed to **[`3.ragas-evaluation.ipynb`](3.ragas-evaluation.ipynb)** to evaluate the RAG system's performance using RAGAS metrics.